In [0]:
import pyspark.sql.functions as F

catalog_name = "f1_dev"
target_table = f"{catalog_name}.silver.dim_drivers"



In [0]:
# 1. Read raw drivers table
df_drivers_raw = spark.table(f"{catalog_name}.bronze.drivers")



In [0]:
# 2. Transform, Cleanse & Anonymize
df_drivers_silver = (
    df_drivers_raw
    # Replace string representation of '\N' with true SQL NULLs
    .replace(r"\N", None)
    # Hash unique driver reference for anonymization/governance
    .withColumn("anonymized_driver_ref", F.sha2(F.col("driverRef"), 256))
    # Concatenate forename and surname into a single full name field
    .withColumn("driver_full_name", F.concat_ws(" ", F.col("forename"), F.col("surname")))
    # Select cleansed schema, deliberately dropping PII columns (dob, url, source)
    .select(
        F.col("driverId").cast("integer").alias("driver_id"),
        F.col("anonymized_driver_ref"),
        F.col("driver_full_name"),
        F.col("code").alias("driver_code"),
        F.col("number").cast("integer").alias("driver_number"),
        F.col("nationality")
    )
)



In [0]:
# 3. Write as Delta table in Silver schema
df_drivers_silver.write.mode("overwrite").saveAsTable(target_table)



In [0]:
# 4. Set Unity Catalog Governance Tags
spark.sql(f"""
    ALTER TABLE {target_table} SET TAGS (
        'layer' = 'silver',
        'table_type' = 'dimension',
        'contains_pii' = 'false_anonymized'
    )
""")

print(f"✓ Successfully built and governed: {target_table}")